# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export — Query ADS API, resolve bibcodes to metadata
2. Translation — Detect languages, translate non-English titles/abstracts
3. Tokenization — Create full_text, tokenize with spaCy
4. AND — Author Name Disambiguation (placeholder)
5. Topic Modeling & Curation — BERTopic + datamapplot + cluster removal
6. Citation Networks — Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import os
from pathlib import Path

# Initialize paths (relative to this notebook's location)
from ads_bib.config import init_paths, load_env
from ads_bib._utils.costs import CostTracker

paths = init_paths()  # uses CWD = notebook directory
load_env()

# Cost tracker for all API calls (OpenRouter, etc.)
tracker = CostTracker()

# Shared OpenRouter pricing mode (used in translation, embeddings, topic labeling)
OPENROUTER_COST_MODE = "hybrid"  # 'hybrid', 'strict', 'fast'

# Verify
print(f"Project root: {paths['project_root']}")
print(f"Data dir:     {paths['data']}")

# If you see a spaCy version warning later, update the model:
# !python -m spacy download en_core_web_lg

Project root: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline
Data dir:     c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\data


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

In [2]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

# Example: Einstein's GR papers + citation expansion + free-text search
Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"  # Einstein GR library
Set_B = f"citations({Set_A}) AND year:1915-2000"   # 1-hop citations
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"  # 2-hop
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"  # Treder library

gr_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gr_terms)}) AND year:1911-2000"
QUERY = f"author:\"Treder, H*\""   # f"({Set_D}) OR ({Set_T}) OR ({Set_E})"

# Set to False to load the latest cached results instead of re-fetching
FORCE_REFRESH = False

### 1.2 Execute Search

In [3]:
from ads_bib.search import search_ads, save_search_results
from ads_bib._utils.io import load_pickle

latest = paths["raw"] / "search_results_latest.pkl"

if FORCE_REFRESH or not latest.exists():
    bibcodes, references, esources, fulltext_urls = search_ads(QUERY, ADS_TOKEN)
    save_search_results((bibcodes, references, esources, fulltext_urls), paths["raw"])
else:
    print(f"Loading cached results: {latest.name}")
    bibcodes, references, esources, fulltext_urls = load_pickle(latest)

print(f"\nBibcodes:        {len(bibcodes):,}")
print(f"Unique refs:     {len(set(r for rl in references for r in rl)):,}")
print(f"Total refs:      {sum(len(rl) for rl in references):,}")

Loading cached results: search_results_latest.pkl

Bibcodes:        382
Unique refs:     499
Total refs:      796


### 1.3 Export & Resolve Metadata

In [4]:
from ads_bib.export import resolve_dataset
from ads_bib._utils.io import save_json_lines, load_json_lines

pubs_path = paths["raw"] / "publications.json"
refs_path = paths["raw"] / "references.json"

if FORCE_REFRESH or not pubs_path.exists():
    publications, refs = resolve_dataset(
        bibcodes, references, esources, fulltext_urls, ADS_TOKEN
    )
    save_json_lines(publications, pubs_path)
    save_json_lines(refs, refs_path)
else:
    print("Loading cached exports ...")
    publications = load_json_lines(pubs_path)
    refs = load_json_lines(refs_path)

print(f"\nPublications: {len(publications):,}")
print(f"References:   {len(refs):,}")
publications.info()

Loading cached exports ...

Publications: 382
References:   499
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 382 entries, 0 to 381
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Bibcode               382 non-null    object
 1   Author                382 non-null    object
 2   Title                 382 non-null    object
 3   Year                  382 non-null    int64 
 4   Journal               382 non-null    object
 5   Journal Abbreviation  382 non-null    object
 6   Issue                 382 non-null    object
 7   Volume                382 non-null    object
 8   First Page            382 non-null    object
 9   Last Page             382 non-null    object
 10  Abstract              382 non-null    object
 11  Keywords              382 non-null    object
 12  DOI                   382 non-null    object
 13  Affiliation           382 non-null    object
 14  Category              382 

In [5]:
publications.head(10)

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,Abstract,Keywords,DOI,Affiliation,Category,Citation Count,References,PDF_URL
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,,,,AA(-),book,52,[],None
1,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,"Wie Infeld 1950 gezeigt hat, ergibt sich aus d...",,10.1002/andp.19574540614,AA(-),article,34,[1953PhRv...92.1567C],None
2,1967AnP...475..194T,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,A theory of the gravitation field is proposed ...,,10.1002/andp.19674750308,AA(-),article,38,"[1916AnP...354..769E, 1940PhRv...57..147R, 195...",None
3,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,The evolution and fundamental questions of phy...,"Astronomical Models, Cosmology, Unified Field ...",10.1002/asna.2113040402,"AA(German Academy of Sciences, Potsdam)",article,1,[1950sts..book.....S],https://ui.adsabs.harvard.edu/link_gateway/198...
4,1972drdt.book.....T,"[Treder, H.J.]",Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,,,,AA(-),book,31,[],None
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,"According to Poincaré, only the ``epistemologi...",,10.1023/A:1018830631884,AA(-); AB(-),article,21,"[1950sts..book.....S, 1971GReGr...2..313T, 199...",None
6,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,,,10.1002/andp.19754870509,AA(-),article,26,"[1963ForPh..11...81T, 1973AnP...485..229T, 197...",None
7,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,,,10.1002/asna.2103020603,AA(-); AB(-),article,6,"[1970AnP...480..315T, 1974AnP...486..325T]",https://ui.adsabs.harvard.edu/link_gateway/198...
8,1988mqg..book.....V,"[von Borzeszkowski, H.H., Treder, H.J.]",The meaning of quantum gravity.,1988,The meaning of quantum gravity.. H.-H. von Bor...,mqg..book,,20,,,Should the general theory of relativity be qua...,,,AA(-); AB(-),book,20,[],None
9,1980AnP...492..250T,"[Treder, H.J.]",Einstein's Hermitian theory of relativity as u...,1980,Annalen der Physik,AnP,,492,250,258,,"Relativistic Astrophysics, Gravitation Theory,...",10.1002/andp.19804920403,AA(-),article,19,"[1957AnP...454..369T, 1975AnP...487..375T]",None


---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

In [6]:
# --- TRANSLATION CONFIGURATION ---
# Provider options: 'openrouter' (API) or 'huggingface' (local, with/without CUDA)

TRANSLATION_PROVIDER = "openrouter"  # or "huggingface"
TRANSLATION_MODEL = "google/gemini-3-flash-preview"  # OpenRouter model, or "google/translategemma-4b-it" for HF
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 50  # parallel workers for API-based translation
# OPENROUTER_COST_MODE is defined in the setup cell above

# Path to fasttext language ID model (download from https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin)
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"


### 2.2 Language Detection

In [7]:
from ads_bib.translate import detect_languages

print("=== Publications ===")
publications = detect_languages(publications, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)

print("\n=== References ===")
refs = detect_languages(refs, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)

=== Publications ===
  Title: 183 non-English entries detected
  Abstract: 35 non-English entries detected

=== References ===
  Title: 185 non-English entries detected
  Abstract: 36 non-English entries detected


### 2.3 Translate Non-English Entries

In [8]:
from ads_bib.translate import translate_dataframe

print("=== Translating Publications ===")
publications, cost_pubs = translate_dataframe(
    publications, ["Title", "Abstract"],
    provider=TRANSLATION_PROVIDER,
    model=TRANSLATION_MODEL,
    api_key=TRANSLATION_API_KEY,
    max_workers=TRANSLATION_MAX_WORKERS,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)

print("\n=== Translating References ===")
refs, cost_refs = translate_dataframe(
    refs, ["Title", "Abstract"],
    provider=TRANSLATION_PROVIDER,
    model=TRANSLATION_MODEL,
    api_key=TRANSLATION_API_KEY,
    max_workers=TRANSLATION_MAX_WORKERS,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)

pub_cost = cost_pubs.get("cost_usd")
ref_cost = cost_refs.get("cost_usd")
print("\nTranslation Cost Snapshot:")
print(f"  Publications: {'n/a' if pub_cost is None else f'${pub_cost:.4f}'}")
print(f"  References:   {'n/a' if ref_cost is None else f'${ref_cost:.4f}'}")


=== Translating Publications ===
  Title: translating 183 entries with openrouter/google/gemini-3-flash-preview ...


  Title: 100%|██████████| 183/183 [00:16<00:00, 11.07it/s]


  Abstract: translating 35 entries with openrouter/google/gemini-3-flash-preview ...


  Abstract: 100%|██████████| 35/35 [00:03<00:00, 10.02it/s]


  Total translation cost: $0.0298 (218/218 priced; direct=218, fetched=0, mode=hybrid)

=== Translating References ===
  Title: translating 185 entries with openrouter/google/gemini-3-flash-preview ...


  Title: 100%|██████████| 185/185 [00:13<00:00, 13.27it/s]


  Abstract: translating 36 entries with openrouter/google/gemini-3-flash-preview ...


  Abstract: 100%|██████████| 36/36 [00:13<00:00,  2.60it/s]

  Total translation cost: $0.0328 (221/221 priced; direct=221, fetched=0, mode=hybrid)

Translation Cost Snapshot:
  Publications: $0.0298
  References:   $0.0328


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize with spaCy.

In [9]:
from ads_bib.tokenize import tokenize_texts

publications = tokenize_texts(publications)
refs = tokenize_texts(refs)

Tokenizing 382 documents with en_core_web_lg ...
  Done.
Tokenizing 499 documents with en_core_web_lg ...
  Done.


### Checkpoint: Save Phase 1-3 Results

In [10]:
from ads_bib._utils.io import save_json_lines

save_json_lines(publications, paths["processed"] / "publications_translated_tokenized.json")
save_json_lines(refs, paths["processed"] / "references_translated_tokenized.json")
print("Checkpoint saved.")

Checkpoint saved.


---
# Phase 4: Author Name Disambiguation (Placeholder)

This step will use an external AND package once it's ready.
For now, the pipeline continues without disambiguation.

In [11]:
# === AND PLACEHOLDER ===
# Uncomment when the AND package is available:
#
# from and_package import disambiguate
# publications = disambiguate(publications)
# refs = disambiguate(refs)

print("AND step skipped (placeholder). Continuing without disambiguation.")

AND step skipped (placeholder). Continuing without disambiguation.


---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

In [12]:
# --- EMBEDDING CONFIGURATION ---
# Providers: 'local', 'huggingface_api', 'openrouter'

EMBEDDING_PROVIDER = "openrouter"
EMBEDDING_MODEL = "google/gemini-embedding-001"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")  # only needed for API providers

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None

### 5.2 Compute Embeddings

In [13]:
from ads_bib._utils.io import load_json_lines

publications = load_json_lines(paths["processed"] / "publications_translated_tokenized.json")
refs = load_json_lines(paths["processed"] / "references_translated_tokenized.json")
print(f"Loaded {len(publications):,} publications")
print(f"Loaded {len(refs):,} references")

Loaded 382 publications
Loaded 499 references


In [14]:
from ads_bib.topic_model import compute_embeddings

df = publications.copy()
if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
    print(f"SAMPLING: {len(df):,} documents")

documents = df["full_text"].tolist()

embeddings = compute_embeddings(
    documents,
    provider=EMBEDDING_PROVIDER,
    model=EMBEDDING_MODEL,
    cache_dir=paths["embeddings_cache"],
    api_key=EMBEDDING_API_KEY,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)
print(f"Embeddings shape: {embeddings.shape}")

  Loaded embeddings from cache: embeddings_openrouter_google_gemini-embedding-001.npz
Embeddings shape: (382, 3072)


### 5.3 Dimensionality Reduction Configuration

**Parameter Guide (PaCMAP & UMAP) for General Relativity and Gravitation (GRG):**

The optimal parameters depend heavily on the size of the dataset:

**1. Small Dataset (e.g., 382 publications)**
- **Reasoning:** The data manifold is very sparse. Using a high `n_neighbors` would force the algorithm to connect completely unrelated subfields, blurring the fine-grained nuances (e.g., separating Black Hole Thermodynamics from Gravitational Wave Data Analysis).
- **PaCMAP (5D/2D):** `PARAMS = dict(n_neighbors=15, MN_ratio=0.5, FP_ratio=2.0, random_state=42)`
  - `MN_ratio` (Mid-Near ratio): Higher values push mid-near points closer. 0.5 is a good balance.
  - `FP_ratio` (Further-Point ratio): Higher values push further points farther away. A value of 2.0 (higher than the default 1.0) helps to separate sparse clusters more distinctly.
- **UMAP (5D for clustering):** `PARAMS_5D = dict(n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42)`
  - `min_dist`: 0.0 allows points to pack tightly, which is ideal for density-based clustering like HDBSCAN.
  - `metric`: "cosine" is standard for high-dimensional text embeddings.
- **UMAP (2D for plotting):** `PARAMS_2D = dict(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)`
  - `min_dist`: 0.1 prevents points from plotting perfectly on top of each other, making the visual map readable.

**2. Large Dataset (e.g., 180,000 publications)**
- **Reasoning:** The GRG space is extremely dense here. A low `n_neighbors` would fragment the data into thousands of noisy micro-clusters. We need a higher value to grasp the massive global structures (Cosmology vs. Formal GR vs. Observational Astrophysics).
- **PaCMAP (5D/2D):** `PARAMS = dict(n_neighbors=60, MN_ratio=0.5, FP_ratio=1.0, random_state=42)`
  - `FP_ratio`: Reduced back to 1.0 because the space is already dense; aggressively pushing points away isn't necessary.
- **UMAP (5D for clustering):** `PARAMS_5D = dict(n_neighbors=50, min_dist=0.0, metric="cosine", random_state=42)`
- **UMAP (2D for plotting):** `PARAMS_2D = dict(n_neighbors=50, min_dist=0.1, metric="cosine", random_state=42)`


In [15]:
# --- DIMENSIONALITY REDUCTION ---
# Methods: 'pacmap' (fast, good) or 'umap' (slower, density-preserving with densmap)
# Note: Toponymy relies heavily on UMAP properties.
DIM_REDUCTION_METHOD = "umap"

# 1. SMALL DATASET (382 docs) PARAMETERS
PARAMS_5D = dict(n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42)
PARAMS_2D = dict(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)

# 2. LARGE DATASET (180k docs) PARAMETERS (Uncomment for full run)
# PARAMS_5D = dict(n_neighbors=50, min_dist=0.0, metric="cosine", random_state=42)
# PARAMS_2D = dict(n_neighbors=50, min_dist=0.1, metric="cosine", random_state=42)

# Cache suffix for uniqueness
model_safe = EMBEDDING_MODEL.replace('/', '_')
CACHE_SUFFIX = f"{EMBEDDING_PROVIDER}_{model_safe}_{DIM_REDUCTION_METHOD}"

In [16]:
from ads_bib.topic_model import reduce_dimensions

reduced_5d, reduced_2d = reduce_dimensions(
    embeddings,
    method=DIM_REDUCTION_METHOD,
    params_5d=PARAMS_5D,
    params_2d=PARAMS_2D,
    cache_dir=paths["dim_reduction_cache"],
    cache_suffix=CACHE_SUFFIX,
)

  Computing 5d_openrouter_google_gemini-embedding-001_umap with UMAP ...


c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.0, n_components=5, n_jobs=1, random_state=42, verbose=True)
Fri Feb 20 09:11:42 2026 Construct fuzzy simplicial set
Fri Feb 20 09:11:42 2026 Finding Nearest Neighbors
Fri Feb 20 09:11:49 2026 Finished Nearest Neighbor Search
Fri Feb 20 09:11:53 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Fri Feb 20 09:11:55 2026 Finished embedding
  Saved: reduced_5d_openrouter_google_gemini-embedding-001_umap.npy
  Computing 2d_openrouter_google_gemini-embedding-001_umap with UMAP ...
UMAP(angular_rp_forest=True, metric='cosine', n_jobs=1, random_state=42, verbose=True)
Fri Feb 20 09:11:55 2026 Construct fuzzy simplicial set


c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Fri Feb 20 09:11:55 2026 Finding Nearest Neighbors
Fri Feb 20 09:11:55 2026 Finished Nearest Neighbor Search
Fri Feb 20 09:11:55 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Fri Feb 20 09:11:56 2026 Finished embedding
  Saved: reduced_2d_openrouter_google_gemini-embedding-001_umap.npy
  5D shape: (382, 5), 2D shape: (382, 2)


### 5.4 Clustering Configuration

**Backend Overview:**
- **bertopic**: Uses HDBSCAN on dimensions reduced by PaCMAP/UMAP. Solid for any dataset size.
- **toponymy**: Creates a hierarchy. Relies on `reduced_5d` (UMAP is recommended over PaCMAP for Toponymy to preserve structures better).
- **toponymy_evoc**: EVoC clustering. Skips the dimensionality reduction step entirely and clusters directly on high-dimensional `embeddings`. Only recommended for large datasets (> 50k documents).

**Parameters Explained:**
- `MIN_CLUSTER_SIZE` (for BERTopic/HDBSCAN & Toponymy): Controls the absolute minimum size a cluster must be to be recognized at the lowest level. We calculate this dynamically (e.g., 0.1% of the dataset) to avoid micro-clusters.
- `min_clusters` (Toponymy only): Forces the hierarchical algorithm to stop merging when it reaches this minimum number of broad topics at the top layer.
- `TOPONYMY_LAYER_INDEX`: Toponymy generates multiple layers of topics (from broad to fine-grained). All layers are saved to the dataset. This parameter simply selects which layer serves as the "primary" layer for baseline metrics and base colors in the visual map (Layer 0 is the finest-grained layer, Layer 1 is broader, etc.).


In [17]:
# --- TOPIC BACKEND + CLUSTERING CONFIGURATION ---
TOPIC_BACKEND = "toponymy_evoc"  # 'bertopic', 'toponymy', 'toponymy_evoc'
CLUSTERING_METHOD = "fast_hdbscan"  # for BERTopic only: 'fast_hdbscan' or 'hdbscan'

# 1. Base Parameter for all backends
n_docs = len(df)
MIN_CLUSTER_SIZE = max(10, int(n_docs * 0.001))  # ~0.1% of dataset
print(f"Dataset: {n_docs:,} documents → min_cluster_size={MIN_CLUSTER_SIZE}")

# 2. BERTopic Parameters (HDBSCAN)
CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=3,
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.02,
)

# 3. Toponymy Parameters
# Selects the primary layer for coloring/base labels (all layers are exported regardless!)
# 0 = finest grained topics, 1 = broader topics, etc.
TOPONYMY_LAYER_INDEX = 0
TOPONYMY_CLUSTER_PARAMS = dict(
    min_clusters=5,  # Stop merging when reaching this many broad topics
    min_samples=3,
    base_min_cluster_size=MIN_CLUSTER_SIZE,  # Lowest level cluster size
)


Dataset: 382 documents → min_cluster_size=10


In [18]:
print(f"Topic backend: {TOPIC_BACKEND}")
print("Clustering configuration is applied directly in the selected topic backend fit function.")


Topic backend: toponymy_evoc
Clustering configuration is applied directly in the selected topic backend fit function.


### 5.5 Topic Modeling Configuration

In [19]:
# --- TOPIC MODELING / LLM LABELING ---
# LLM Providers: 'local', 'huggingface_api', 'openrouter'

LLM_PROVIDER = "openrouter"
LLM_MODEL = "google/gemini-3-flash-preview"  # or an OpenRouter model
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Representation pipeline (BERTopic only)
PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

# Toponymy text embedder model (OpenRouter/OpenAI-compatible endpoint)
TOPONYMY_EMBEDDING_MODEL = EMBEDDING_MODEL

# min_df for CountVectorizer: use 1 for small samples, 2+ for full data
MIN_DF = 1 if (SAMPLE_SIZE and SAMPLE_SIZE < 1000) else 2


In [20]:
from ads_bib.topic_model import fit_bertopic, fit_toponymy, reduce_outliers, build_topic_dataframe
import numpy as np

if TOPIC_BACKEND == "bertopic":
    topic_model = fit_bertopic(
        documents,
        reduced_5d,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        pipeline_models=PIPELINE_MODELS,
        parallel_models=PARALLEL_MODELS,
        min_df=MIN_DF,
        clustering_method=CLUSTERING_METHOD,
        clustering_params=CLUSTER_PARAMS,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    topics = np.array(topic_model.topics_)
    topics = reduce_outliers(
        topic_model,
        documents,
        topics,
        reduced_5d,
        threshold=0.8,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
    topic_info = topic_model.get_topic_info()

elif TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    topic_model, topics, topic_info = fit_toponymy(
        documents,
        embeddings,
        reduced_5d,
        backend=TOPIC_BACKEND,
        layer_index=TOPONYMY_LAYER_INDEX,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        embedding_model=TOPONYMY_EMBEDDING_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        clusterer_params=TOPONYMY_CLUSTER_PARAMS,
        cost_tracker=tracker,
    )

else:
    raise ValueError(
        f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'. "
        "Use 'bertopic', 'toponymy', or 'toponymy_evoc'."
    )

df = build_topic_dataframe(
    df,
    topic_model,
    topics,
    reduced_2d,
    embeddings,
    topic_info=topic_info,
)
topic_info


  Using raw embeddings for clustering with EVoCClusterer.
Fitting Toponymy backend='toponymy_evoc' (LLM: openrouter/google/gemini-3-flash-preview) ...


Selecting facility_location exemplars:   0%|          | 0/16 [00:00<?, ?cluster/s]

embedding texts:   0%|          | 0/27 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/2 [00:00<?, ?layer/s]

Generating informative keyphrases:   0%|          | 0/16 [00:00<?, ?cluster/s]

...

Selecting facility_location subtopics:   0%|          | 0/7 [00:00<?, ?cluster/s]

Generating prompts for layer 1:   0%|          | 0/7 [00:00<?, ?topic/s]

Generating topic names for layer 1:   0%|          | 0/7 [00:00<?, ?topic/s]

  Selected Toponymy layer 0/1: 16 topics, 120 outliers.
  OpenRouter cost: $0.0101 (20/20 priced; direct=20, fetched=0, mode=hybrid)


,Topic,Name,Main
0,-1,Outlier Topic,Outlier Topic
1,0,Relativistic Generalizations of Ertel's and He...,Relativistic Generalizations of Ertel's and He...
2,1,"Singularities, Shock Waves, and Particle Model...","Singularities, Shock Waves, and Particle Model..."
3,2,"Fundamental Physical Constants, Planckions, an...","Fundamental Physical Constants, Planckions, an..."
4,3,Foundational Intersections of Quantum Mechanic...,Foundational Intersections of Quantum Mechanic...
5,4,"Biographical Tributes, Obituaries, and Histori...","Biographical Tributes, Obituaries, and Histori..."
6,5,Historical Perspectives on Geophysical Science...,Historical Perspectives on Geophysical Science...
7,6,Mach's Principle and Mach-Einstein Doctrine in...,Mach's Principle and Mach-Einstein Doctrine in...
8,7,Analytical Formulations of the Mach-Einstein D...,Analytical Formulations of the Mach-Einstein D...
9,8,Lorentz-Covariant Tetrad Theories and Generali...,Lorentz-Covariant Tetrad Theories and Generali...


### 5.6 Visualize Topics

In [21]:
from ads_bib.visualize import create_topic_map

# Use all layers for Toponymy, or just 'Name' for BERTopic
if TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    num_layers = len([col for col in df.columns if col.startswith("Topic_Layer_")])
    label_column = [f"Topic_Layer_{i}" for i in range(num_layers)]
else:
    label_column = "Name"

plot = create_topic_map(
    df,
    label_column=label_column,
    title="ADS Bibliometric Map",
    subtitle=f"Topics labeled with {LLM_PROVIDER}/{LLM_MODEL}",
    dark_mode=True,
    output_path=paths["output"] / "topic_map.html",
)
# plot  # uncomment to display inline

Saved: topic_map.html


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [22]:
from ads_bib.curate import get_cluster_summary, remove_clusters

get_cluster_summary(df)

,topic_id,Count,Percentage,Label
0,-1,120,31.4,Outlier Topic
3,2,24,6.3,"Fundamental Physical Constants, Planckions, an..."
9,8,24,6.3,Lorentz-Covariant Tetrad Theories and Generali...
10,9,24,6.3,"Unified Field Theory, Purely Affine Gravity, a..."
6,5,20,5.2,Historical Perspectives on Geophysical Science...
1,0,20,5.2,Relativistic Generalizations of Ertel's and He...
2,1,19,5.0,"Singularities, Shock Waves, and Particle Model..."
13,12,18,4.7,Theoretical Foundations of Early Universe Evol...
14,13,16,4.2,Historical Evolution of Celestial Mechanics an...
4,3,15,3.9,Foundational Intersections of Quantum Mechanic...


In [23]:
# Remove unwanted clusters (set IDs based on the summary above)
CLUSTERS_TO_REMOVE = []  # e.g. [3, 7, -1]

if CLUSTERS_TO_REMOVE:
    df = remove_clusters(df, CLUSTERS_TO_REMOVE)

### Checkpoint: Save Curated Dataset

In [24]:
from ads_bib._utils.io import save_parquet

# Ensure References is list type for Parquet
df["References"] = df["References"].apply(lambda x: x if isinstance(x, list) else [])

save_parquet(df, paths["processed"] / "curated_dataset.parquet")
print(f"Curated dataset: {len(df):,} documents")

Curated dataset: 382 documents


In [25]:
from ads_bib._utils.io import load_parquet

df = load_parquet(paths["processed"] / "curated_dataset.parquet")
df.head(10)

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,full_text,tokens,embedding_2d_x,embedding_2d_y,topic_id,Name,Main,full_embeddings,Topic_Layer_0,Topic_Layer_1
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,...,Theory of Gravitation and Principle of Equival...,"[theory, gravitation, principle, equivalence]",12.903994,8.045989,14,Experimental Foundations of General Relativity...,Experimental Foundations of General Relativity...,"[0.00079854747, -0.019891707, 0.020806318, -0....",Experimental Foundations of General Relativity...,Relativistic Gravitation
1,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,...,Definition of Electric Charge and Electric For...,"[definition, electric, charge, electric, force...",14.237543,6.162948,1,"Singularities, Shock Waves, and Particle Model...","Singularities, Shock Waves, and Particle Model...","[-0.010657256, -0.003516388, 0.025264107, -0.0...","Singularities, Shock Waves, and Particle Model...",Unlabelled
2,1967AnP...475..194T,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,...,The Macroscopic Gravitational Field in the Uni...,"[macroscopic, gravitational, field, unified, q...",13.583593,5.302496,9,"Unified Field Theory, Purely Affine Gravity, a...","Unified Field Theory, Purely Affine Gravity, a...","[0.0009005609, -0.020405725, 0.010572005, -0.0...","Unified Field Theory, Purely Affine Gravity, a...","Unified Field Theory, Purely Affine Gravity, a..."
3,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,...,The problem of the Grand Unification Theory. T...,"[problem, grand, unification, theory, evolutio...",12.789717,5.674225,-1,Outlier Topic,Outlier Topic,"[-0.014908035, -0.018396216, 0.014744086, -0.0...",Unlabelled,Unlabelled
4,1972drdt.book.....T,"[Treder, H.J.]",Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,...,The Relativity of Inertia..,"[relativity, inertia]",13.631322,8.390042,-1,Outlier Topic,Outlier Topic,"[0.022441277, -0.018025622, 0.012645888, -0.08...",Unlabelled,Unlabelled
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,...,The Weyl-Cartan Space Problem in Purely Affine...,"[weyl, cartan, space, problem, purely, affine,...",13.896609,4.574677,9,"Unified Field Theory, Purely Affine Gravity, a...","Unified Field Theory, Purely Affine Gravity, a...","[-0.020331278, -0.015941117, 0.03069637, -0.06...","Unified Field Theory, Purely Affine Gravity, a...","Unified Field Theory, Purely Affine Gravity, a..."
6,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,...,On the unitarized theory of gravitation with l...,"[unitarize, theory, gravitation, short, range,...",13.163076,6.860907,-1,Outlier Topic,Outlier Topic,"[-0.0044219275, -0.009710963, 0.009115518, -0....",Unlabelled,Relativistic Gravitation
7,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,...,On Soldners Value of Newtonian Deflection of L...,"[soldners, value, newtonian, deflection, light]",12.283854,8.776442,10,Relativistic Gravito-Optical Effects and Anoma...,Relativistic Gravito-Optical Effects and Anoma...,"[0.014637167, -0.025634238, 0.013352524, -0.07...",Relativistic Gravito-Optical Effects and Anoma...,Relativistic Gravitation
8,1988mqg..book.....V,"[von Borzeszkowski, H.H., Treder, H.J.]",The meaning of quantum gravity.,1988,The meaning of quantum gravity.. H.-H. von Bor...,mqg..book,,20,,,...,The meaning of quantum gravi

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

In [26]:
# --- CITATION CONFIGURATION ---
METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 1,
    "co_citation": 1,
    "bibliographic_coupling": 1,
    "author_co_citation": 1,
}
AUTHORS_FILTER = None  # e.g. ['Treder', 'Kreisel'] or None for all
OUTPUT_FORMAT = "gexf"  # Options: 'gexf', 'graphology', 'csv', 'all'

### 6.2 Build Node Table & Compute Networks

In [27]:
from ads_bib.citations import build_all_nodes, process_all_citations, export_wos_format

# Reload processed data if needed
all_nodes = build_all_nodes(publications, refs)

results = process_all_citations(
    bibcodes, references, publications, refs, all_nodes,
    metrics=METRICS,
    min_counts=MIN_COUNTS,
    authors_filter=AUTHORS_FILTER,
    output_format=OUTPUT_FORMAT,
    output_dir=paths["output"],
)

All nodes: 773
Processing direct ...
  GEXF: direct.gexf
Processing co_citation ...
  GEXF: co_citation.gexf
Processing bibliographic_coupling ...
  GEXF: bibliographic_coupling.gexf
Processing author_co_citation ...
  GEXF: author_co_citation.gexf


### 6.3 Export Web of Science Format

In [28]:
suffix = "_filtered" if AUTHORS_FILTER else ""
export_wos_format(
    publications, refs,
    output_path=paths["output"] / f"download_wos_export{suffix}.txt",
)

  WOS format: download_wos_export.txt


---
# Summary

Final dataset statistics and output file listing.

In [29]:
import os

print("="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Publications:     {len(publications):,}")
print(f"References:       {len(refs):,}")
print(f"Curated dataset:  {len(df):,}")
print(f"Topics found:     {df['topic_id'].nunique()}")
print()
print("Output files:")
for root, dirs, files in os.walk(paths["output"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f"  {fpath.relative_to(paths['output'])} ({size_mb:.1f} MB)")

print()
print(tracker.compact_summary())


PIPELINE COMPLETE
Publications:     382
References:       499
Curated dataset:  382
Topics found:     17

Output files:
  BERTopic_topic_map.html (0.2 MB)
  Toponomy_Evoc_topic_map.html (0.3 MB)
  Toponomy_topic_map.html (0.2 MB)
  author_co_citation.gexf (0.7 MB)
  bibliographic_coupling.gexf (0.8 MB)
  co_citation.gexf (2.1 MB)
  direct.gexf (1.6 MB)
  download_wos_export.txt (0.2 MB)
  topic_map.html (0.3 MB)

CostTracker Summary (compact)
  translation | google/gemini-3-flash-preview | tokens(total=51,756, prompt=37,060, completion=14,696) | calls=2 | cost=$0.0626
  llm_labeling_toponymy_evoc | google/gemini-3-flash-preview | tokens(total=16,501, prompt=15,779, completion=722) | calls=1 | cost=$0.0101
  TOTAL | all models | tokens(total=68,257, prompt=-, completion=-) | calls=3 | cost=$0.0727
